# SCA2 Tier-2 Evaluation Surfaces
## Merge quality · exploratory stats · colleague usage recipe

**Purpose**

1. **QA** the four merged parquet files (schema, coverage, weights, missingness).
2. **Compute human country moments** for a pre-registered Tier-2 benchmark skeleton.
3. **Show a colleague how to load files from Google Drive** and prepare them for **frozen** DPO adapter scoring (**no retraining**).

**Docs in this folder**

| File | Role |
|------|------|
| `DATASET_GUIDE.md` | Selection criteria, exact item wording, limitations |
| `CONSTRUCT_MAP.md` | Compact GPS → WVS / AB construct map |
| `_manifest.json` | Per-wave machine coverage |

**Non-claims**

- Matching human **signs** is directional evidence, not GPS magnitude recovery.
- AmericasBarometer is a **trust / political-culture** surface — not a six-preference GPS battery.
- Do **not** retrain adapters on these surveys.


## 0. Setup

Point `DATA` at the folder that contains the four `.parquet` files (this notebook's directory if you unzipped the Drive folder as-is).

Requires: `pandas`, `pyarrow`. Optional for plots: `matplotlib`, `seaborn`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

HAS_PLOTS = False
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    try:
        import seaborn as sns
        sns.set_theme(style="whitegrid", context="notebook")
    except Exception:
        sns = None
    HAS_PLOTS = True
except Exception as e:
    print(f"Plotting unavailable ({type(e).__name__}: {e}). Tables only.")
    plt = None
    sns = None

# --- SET THIS when loading from Google Drive ---
DATA = Path(".").resolve()
if not (DATA / "USA_WVS_wave7.parquet").exists():
    alt = Path("/Users/bonorinoa/Desktop/Github_Repositories/SCA2_PofW/data/merged")
    if (alt / "USA_WVS_wave7.parquet").exists():
        DATA = alt
print("DATA =", DATA)
print("HAS_PLOTS =", HAS_PLOTS)


## 1. Load the four evaluation surfaces

Each file is one country × one survey. **Do not** row-stack WVS with AmericasBarometer.


In [ ]:
FILES = {
    "USA_WVS": "USA_WVS_wave7.parquet",
    "MEX_WVS": "MEX_WVS_wave7.parquet",
    "USA_AB": "USA_Barometer_2012_2019.parquet",
    "MEX_AB": "MEX_Barometer_2012_2019.parquet",
}
dfs = {k: pd.read_parquet(DATA / v) for k, v in FILES.items()}
for k, df in dfs.items():
    print(f"{k:8s}  n={len(df):5d}  cols={df.shape[1]:3d}  "
          f"country={df['country'].unique().tolist()}  "
          f"years={sorted(df['year'].dropna().astype(int).unique().tolist())}")


## 2. Schema & provenance check


In [ ]:
PROVENANCE = ["survey", "country", "year", "source_file"]
rows = []
for name, df in dfs.items():
    rows.append({
        "dataset": name,
        "n": len(df),
        "ncols": df.shape[1],
        "years": sorted(df["year"].dropna().astype(int).unique().tolist()),
        "n_by_year": {int(k): int(v) for k, v in df.groupby("year").size().items()},
        "has_provenance": all(c in df.columns for c in PROVENANCE),
        "weight_nonnull": float(df["weight"].notna().mean()) if "weight" in df.columns else None,
        "weight_mean": float(pd.to_numeric(df["weight"], errors="coerce").mean()) if "weight" in df.columns else None,
    })
overview = pd.DataFrame(rows)
display(overview)
print("\nWVS USA columns:", list(dfs["USA_WVS"].columns))
print("\nAB USA columns:", list(dfs["USA_AB"].columns))


## 3. Missing-code convention

WVS / LAPOP store DK/NA as special codes (88, 98, long 888888 codes, or negative WVS codes).
**Always mask before means.** Values are **raw** — no reverse-coding at merge time.


In [ ]:
MISS_CODES = {88, 98, 888888, 988888, 999999, -1, -2, -3, -4, -5}

def mask_miss(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    return x.where(~x.isin(MISS_CODES))

for name, items in [
    ("USA_WVS", ["Q57", "Q106", "Q177"]),
    ("USA_AB", ["it1", "b4", "exc7"]),
]:
    df = dfs[name]
    for c in items:
        if c not in df.columns:
            continue
        raw = pd.to_numeric(df[c], errors="coerce")
        print(f"{name}.{c}: missing-code share={float(raw.isin(MISS_CODES).mean()):.3f}, "
              f"valid n={mask_miss(df[c]).notna().sum()}")


## 4. AmericasBarometer — trust-core coverage by year

USA 2017 is WVS-adjacent but has a **thinner** core battery.


In [ ]:
AB_CORE = [
    "it1", "b1", "b2", "b3", "b4", "b6",
    "b10a", "b12", "b13", "b18", "b21", "b31", "b32", "b37", "b47a",
    "exc6", "exc7", "ing4", "pn4", "dem2",
]
cov_rows = []
for key, ctry in [("USA_AB", "USA"), ("MEX_AB", "MEX")]:
    df = dfs[key]
    for year, g in df.groupby("year"):
        row = {"country": ctry, "year": int(year), "n": len(g)}
        present = 0
        for c in AB_CORE:
            if c in g.columns:
                rate = float(mask_miss(g[c]).notna().mean())
                row[c] = round(rate, 3)
                if rate > 0:
                    present += 1
            else:
                row[c] = None
        row["n_core_with_data"] = present
        cov_rows.append(row)
ab_cov = pd.DataFrame(cov_rows)
show = ["country", "year", "n", "n_core_with_data",
        "it1", "b1", "b2", "b4", "b6", "ing4", "pn4", "exc7", "dem2", "b10a"]
display(ab_cov[show])

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, ctry in zip(axes, ["USA", "MEX"]):
        sub = ab_cov[ab_cov["country"] == ctry]
        ax.bar(sub["year"].astype(str), sub["n_core_with_data"],
               color="#2c7fb8" if ctry == "USA" else "#d95f0e")
        ax.set_title(f"{ctry} AB: # core items with valid data")
        ax.set_xlabel("year")
        ax.set_ylabel("# items (of 20)")
        ax.set_ylim(0, 21)
        ax.axhline(20, color="gray", ls="--", lw=0.8)
    plt.tight_layout()
    plt.show()


## 5. WVS — mapped item coverage


In [ ]:
WVS_TIER2 = {
    "trust": ["Q57", "Q59", "Q61", "Q62", "Q63", "Q64", "Q69", "Q70", "Q71"],
    "patience": ["Q13", "Q14", "Q43"],
    "risktaking": ["Q106", "Q107", "Q109"],
    "posrecip": ["Q12", "Q174"],
    "negrecip": ["Q176", "Q177", "Q179"],
    "altruism": ["Q101", "Q99"],
}
WVS_TIER3 = {
    "trust": ["Q58", "Q60", "Q73"],
    "patience": ["Q50"],
    "risktaking": ["Q178"],
    "posrecip": ["Q81"],
    "negrecip": ["Q195"],
    "altruism": ["Q103"],
}
WVS_ITEMS = sorted({i for d in (WVS_TIER2, WVS_TIER3) for items in d.values() for i in items})

wvs_nn_rows = []
for key, ctry in [("USA_WVS", "USA"), ("MEX_WVS", "MEX")]:
    df = dfs[key]
    for q in WVS_ITEMS:
        rate = float(mask_miss(df[q]).notna().mean()) if q in df.columns else 0.0
        wvs_nn_rows.append({"country": ctry, "item": q, "nonnull": rate, "present": q in df.columns})
wvs_nn = pd.DataFrame(wvs_nn_rows)
missing = wvs_nn.loc[~wvs_nn["present"], ["country", "item"]]
print("Missing columns:", "None" if missing.empty else missing.to_string(index=False))
display(wvs_nn.pivot(index="item", columns="country", values="nonnull").round(3))


## 6. Human moments — WVS (unweighted)

**Polarity caution:** raw scales differ. Example: Q57 lower mean ≈ *higher* generalized trust.
Full wording: `DATASET_GUIDE.md`.


In [ ]:
FOCUS_WVS = [
    "Q57", "Q59", "Q61", "Q62", "Q63", "Q69", "Q70", "Q71",
    "Q13", "Q14", "Q106", "Q107", "Q109",
    "Q176", "Q177", "Q179", "Q101", "Q99",
]
POLARITY_WVS = {
    "Q57": "lower = more generalized trust (1 trust, 2 careful)",
    "Q59": "lower = more trust (1 complete … 4 not at all)",
    "Q61": "lower = more trust strangers",
    "Q62": "lower = more trust other religion",
    "Q63": "lower = more trust other nationality",
    "Q69": "lower = more confidence in police",
    "Q70": "lower = more confidence in courts",
    "Q71": "lower = more confidence in government",
    "Q13": "higher = thrift mentioned more",
    "Q14": "higher = perseverance mentioned more",
    "Q106": "higher = more pro-incentives / less equality",
    "Q107": "higher = more pro-government ownership",
    "Q109": "higher = competition more harmful",
    "Q176": "higher = more disagreement that moral rules are unclear",
    "Q177": "higher = more justifiable unearned benefits",
    "Q179": "higher = more justifiable stealing",
    "Q101": "higher = more charity org membership",
    "Q99": "higher = more environmental org membership",
}
rows = []
for q in FOCUS_WVS:
    usa = mask_miss(dfs["USA_WVS"][q])
    mex = mask_miss(dfs["MEX_WVS"][q])
    mu, mm = float(usa.mean()), float(mex.mean())
    rows.append({
        "item": q,
        "USA_mean": round(mu, 3),
        "MEX_mean": round(mm, 3),
        "diff_USA_minus_MEX": round(mu - mm, 3),
        "USA_n": int(usa.notna().sum()),
        "MEX_n": int(mex.notna().sum()),
        "polarity_note": POLARITY_WVS.get(q, ""),
    })
wvs_moments = pd.DataFrame(rows)
display(wvs_moments)

if HAS_PLOTS:
    plot_items = ["Q57", "Q61", "Q63", "Q109", "Q177"]
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(plot_items))
    usa_m = [wvs_moments.loc[wvs_moments.item == q, "USA_mean"].iloc[0] for q in plot_items]
    mex_m = [wvs_moments.loc[wvs_moments.item == q, "MEX_mean"].iloc[0] for q in plot_items]
    ax.bar(x - 0.18, usa_m, 0.35, label="USA", color="#2c7fb8")
    ax.bar(x + 0.18, mex_m, 0.35, label="MEX", color="#d95f0e")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_items)
    ax.set_ylabel("Unweighted mean (raw codes)")
    ax.set_title("WVS selected items — USA vs MEX (raw scale)")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 7. Human moments — AmericasBarometer (pooled 2012–2019)

Primary AB surface: **IT1**, system support **B1–B6**, **ING4/PN4**, **EXC** when available.


In [ ]:
FOCUS_AB = ["it1", "b1", "b2", "b3", "b4", "b6", "ing4", "pn4", "exc7"]
POLARITY_AB = {
    "it1": "lower = more community trust (1 very trustworthy … 4 untrustworthy)",
    "b1": "higher = courts guarantee fair trial more",
    "b2": "higher = more respect for institutions",
    "b3": "higher = rights better protected",
    "b4": "higher = prouder of political system",
    "b6": "higher = should support system more",
    "ing4": "higher = more agree democracy best form",
    "pn4": "lower = more satisfied with democracy",
    "exc7": "lower = corruption seen as more common (1 very common … 4 very uncommon)",
}
rows = []
for c in FOCUS_AB:
    usa = mask_miss(dfs["USA_AB"][c]) if c in dfs["USA_AB"].columns else pd.Series(dtype=float)
    mex = mask_miss(dfs["MEX_AB"][c]) if c in dfs["MEX_AB"].columns else pd.Series(dtype=float)
    mu = float(usa.mean()) if usa.notna().any() else np.nan
    mm = float(mex.mean()) if mex.notna().any() else np.nan
    rows.append({
        "item": c,
        "USA_mean": None if np.isnan(mu) else round(mu, 3),
        "MEX_mean": None if np.isnan(mm) else round(mm, 3),
        "diff_USA_minus_MEX": None if (np.isnan(mu) or np.isnan(mm)) else round(mu - mm, 3),
        "USA_n": int(usa.notna().sum()),
        "MEX_n": int(mex.notna().sum()),
        "polarity_note": POLARITY_AB.get(c, ""),
    })
ab_moments = pd.DataFrame(rows)
display(ab_moments)

print("\nBy-year means:")
for c in ["it1", "b4", "ing4", "pn4"]:
    print(f"\n{c}:")
    for key, ctry in [("USA_AB", "USA"), ("MEX_AB", "MEX")]:
        df = dfs[key]
        if c not in df.columns:
            print(f"  {ctry}: column absent")
            continue
        series = df.groupby("year").apply(lambda g: mask_miss(g[c]).mean(), include_groups=False)
        print(f"  {ctry}: " + ", ".join(f"{int(y)}={v:.3f}" for y, v in series.items() if pd.notna(v)))

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(8, 4))
    for key, ctry, color in [("USA_AB", "USA", "#2c7fb8"), ("MEX_AB", "MEX", "#d95f0e")]:
        df = dfs[key]
        s = df.groupby("year").apply(lambda g: mask_miss(g["it1"]).mean(), include_groups=False)
        ax.plot(s.index.astype(int), s.values, marker="o", label=ctry, color=color)
    ax.set_xlabel("year")
    ax.set_ylabel("IT1 mean (lower = more trust)")
    ax.set_title("AmericasBarometer IT1 by year")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 8. Colleague recipe — load from Drive & human benchmark table


In [ ]:
def load_tier2_surfaces(data_dir: str | Path) -> dict[str, pd.DataFrame]:
    """Load the four SCA2 tier-2 evaluation parquets from a Drive download."""
    data_dir = Path(data_dir)
    return {
        "USA_WVS": pd.read_parquet(data_dir / "USA_WVS_wave7.parquet"),
        "MEX_WVS": pd.read_parquet(data_dir / "MEX_WVS_wave7.parquet"),
        "USA_AB": pd.read_parquet(data_dir / "USA_Barometer_2012_2019.parquet"),
        "MEX_AB": pd.read_parquet(data_dir / "MEX_Barometer_2012_2019.parquet"),
    }


def human_moment_table(usa: pd.DataFrame, mex: pd.DataFrame, items: list[str]) -> pd.DataFrame:
    """Unweighted means by country (missing codes masked)."""
    rows = []
    for q in items:
        if q not in usa.columns or q not in mex.columns:
            rows.append({
                "item": q, "USA_mean": None, "MEX_mean": None,
                "diff_USA_minus_MEX": None, "USA_n": 0, "MEX_n": 0, "status": "missing_column",
            })
            continue
        u, m = mask_miss(usa[q]), mask_miss(mex[q])
        mu, mm = float(u.mean()), float(m.mean())
        rows.append({
            "item": q,
            "USA_mean": mu,
            "MEX_mean": mm,
            "diff_USA_minus_MEX": mu - mm,
            "USA_n": int(u.notna().sum()),
            "MEX_n": int(m.notna().sum()),
            "status": "ok",
        })
    return pd.DataFrame(rows)


benchmark_wvs = human_moment_table(
    dfs["USA_WVS"], dfs["MEX_WVS"],
    ["Q57", "Q61", "Q63", "Q70", "Q106", "Q109", "Q177", "Q101"],
)
benchmark_ab = human_moment_table(
    dfs["USA_AB"], dfs["MEX_AB"],
    ["it1", "b2", "b4", "b6", "ing4", "pn4"],
)
print("WVS benchmark skeleton")
display(benchmark_wvs.round(3))
print("AB benchmark skeleton")
display(benchmark_ab.round(3))


## 9. Adapter scoring protocol (outline — no model calls here)

For each mapped item:

1. Prompt with **official wording** from `DATASET_GUIDE.md`.
2. Offer the same response options (or a forced ranking consistent with DPO).
3. Score **both** USA and MEX adapters (optionally base reference).
4. Compare own-country alignment and cross-country specialization.
5. Report by GPS dimension with tags: `clean | bridge | stretch | no-coverage`.

**Do not retrain.** Suggested first batch: WVS Q57/Q61/Q63; AB IT1/B2/B4/B6; then bridge items Q106/Q109/Q177.


## 10. QA checklist


In [ ]:
checks = []

def check(name, cond, detail=""):
    checks.append({"check": name, "pass": bool(cond), "detail": detail})
    print(("PASS" if cond else "FAIL"), "-", name, detail)

check("USA_WVS n≈2596", abs(len(dfs["USA_WVS"]) - 2596) < 5, f"n={len(dfs['USA_WVS'])}")
check("MEX_WVS n≈1741", abs(len(dfs["MEX_WVS"]) - 1741) < 5, f"n={len(dfs['MEX_WVS'])}")
check("USA_AB n=6000", len(dfs["USA_AB"]) == 6000, f"n={len(dfs['USA_AB'])}")
check("MEX_AB n=6238", len(dfs["MEX_AB"]) == 6238, f"n={len(dfs['MEX_AB'])}")
check("USA_AB years", sorted(dfs["USA_AB"]["year"].unique()) == [2012, 2014, 2017, 2019], "")
check("MEX_AB years", sorted(dfs["MEX_AB"]["year"].unique()) == [2012, 2014, 2017, 2019], "")
check("WVS items present USA", all(q in dfs["USA_WVS"].columns for q in WVS_ITEMS), "")
check("AB it1 present both", "it1" in dfs["USA_AB"].columns and "it1" in dfs["MEX_AB"].columns, "")
check("weights nonnull WVS USA", dfs["USA_WVS"]["weight"].notna().mean() > 0.99, "")
check("weights nonnull AB USA", dfs["USA_AB"]["weight"].notna().mean() > 0.99, "")
print("\nSummary:", sum(c["pass"] for c in checks), "/", len(checks), "passed")


## 11. Notes

- Prefer **unweighted** moments until weights are re-validated.
- USA AB **2017** is thinner — prefer IT1 + B2/B3/B4/B6 + ING4 for year-adjacent comparisons.
- Full wording: **`DATASET_GUIDE.md`**. Lab rebuild: `_build_merge.py`.
